# Research foundations: what do my saved order books tell me?

**Question:** what visible liquidity did OKX and Bitfinex report, and how confidently
can I reconstruct it from saved inputs?

This notebook reads **OKX BTC-USDT historical L2 books** (aggregate price levels)
and **Bitfinex BTC/USD recorded live R0 books** (visible individual orders).
It uses the existing package readers. It makes **no network requests** and needs
no account or credentials. To obtain inputs first, run
[market-data-capture.ipynb](market-data-capture.ipynb), which records public feeds
and downloads the selected OKX day.

These are order books, not trades. They do not replace the
[frozen P1 trade study](../docs/research/p1-baseline.md). The sources have different
quote currencies, observation times and depth limits; their prices are not a
simultaneous arbitrage comparison. Restart the kernel and Run All in order.


## 1. Which implementation am I using?

The checks below require the pinned interpreter and an installed package whose
runtime files match this checkout. The table identifies this execution so that a
later result can be compared with the same code and saved inputs.


In [1]:
# standard library imports
import hashlib
import json
import os
import platform
import subprocess
import sys
from datetime import UTC, datetime, timedelta
from decimal import Decimal
from html import escape
from importlib.metadata import distributions
from pathlib import Path

# third party library imports
from IPython.display import HTML, display

# project library imports
import alpha_research
from alpha_research.data.live.config import CaptureConfig, CaptureRequest
from alpha_research.data.live.journal import CaptureJournal, CaptureRecord
from alpha_research.data.live.replay import CaptureReplay
from alpha_research.data.raw_objects import RawObject
from alpha_research.data.venues.bitfinex.book import BitfinexBook
from alpha_research.data.venues.okx.history_config import OKXArchiveRequest, OKXHistoryConfig
from alpha_research.data.venues.okx.history_store import OKXArchiveStore
from alpha_research.serialization import DomainCodec

root = Path.cwd().resolve()
if root.name == "examples":
    root = root.parent
assert (root / "configs/history/okx-btc-usdt.json").is_file(), (
    "Run from the repository root or examples/."
)
notebook_path = Path(
    os.environ.get("ALPHA_RESEARCH_NOTEBOOK_PATH", root / "examples/research-foundations.ipynb")
).resolve()


class NotebookDisplay:
    """Render small tables for the exploratory notebook scenarios."""

    @staticmethod
    def show_table(headers: tuple[str, ...], rows: list[tuple[object, ...]]) -> None:
        """Display a small escaped table without adding a dataframe dependency.

        Args:
            headers: Column labels.
            rows: Exact or presentation values to render as text.
        """
        heading = "".join(
            f'<th style="text-align:left;padding:6px">{escape(label)}</th>' for label in headers
        )
        body = "".join(
            "<tr>"
            + "".join(
                f'<td style="padding:6px;vertical-align:top">{escape(str(value))}</td>'
                for value in row
            )
            + "</tr>"
            for row in rows
        )
        display(HTML(f"<table><thead><tr>{heading}</tr></thead><tbody>{body}</tbody></table>"))

In [2]:
assert platform.python_version() == (root / ".python-version").read_text().strip() == "3.14.7"
installed = Path(alpha_research.__file__).resolve().parent
assert "site-packages" in installed.parts and installed != root / "src/alpha_research"
source_root = root / "src/alpha_research"
runtime_files = sorted(
    path
    for path in source_root.rglob("*")
    if path.is_file() and path.suffix in {".py", ".json", ".typed"}
)
assert runtime_files
runtime_digest = hashlib.sha256()
for source_file in runtime_files:
    relative = source_file.relative_to(source_root)
    source_bytes = source_file.read_bytes()
    assert source_bytes == (installed / relative).read_bytes(), f"Stale installed file: {relative}"
    runtime_digest.update(str(relative).encode() + b"\0" + source_bytes + b"\0")

commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=root, text=True).strip()
branch = subprocess.check_output(["git", "branch", "--show-current"], cwd=root, text=True).strip()
working_status = subprocess.check_output(["git", "status", "--short"], cwd=root, text=True).strip()
candidate_diff = subprocess.check_output(
    [
        "git",
        "diff",
        "--binary",
        "HEAD",
        "--",
        "src",
        "configs",
        "pyproject.toml",
        "uv.lock",
        ".python-version",
    ],
    cwd=root,
)
notebook_document = json.loads(notebook_path.read_text())
notebook_source = [
    (cell["cell_type"], "".join(cell["source"])) for cell in notebook_document["cells"]
]
notebook_digest = hashlib.sha256(
    json.dumps(notebook_source, ensure_ascii=False, separators=(",", ":")).encode()
).hexdigest()
identity = {
    "notebook_path": str(notebook_path),
    "attempt_id": os.environ.get("ALPHA_RESEARCH_ATTEMPT_ID", "interactive: unsnapshotted"),
    "commit": commit,
    "branch": branch,
    "working_tree": "changed" if working_status else "clean",
    "python": platform.python_version(),
    "platform": platform.platform(),
    "kernel": sys.executable,
    "package": str(installed),
    "verified_runtime_files": len(runtime_files),
    "runtime_sha256": runtime_digest.hexdigest(),
    "runtime_input_diff_sha256": hashlib.sha256(candidate_diff).hexdigest(),
    "uv_lock_sha256": hashlib.sha256((root / "uv.lock").read_bytes()).hexdigest(),
    "okx_config_sha256": hashlib.sha256(
        (root / "configs/history/okx-btc-usdt.json").read_bytes()
    ).hexdigest(),
    "notebook_source_sha256": notebook_digest,
    "notebook_tooling": {
        item.metadata["Name"]: item.version
        for item in distributions()
        if item.metadata["Name"] in {"nbclient", "nbformat", "ipykernel"}
    },
}
NotebookDisplay.show_table(("Identity", "Observed value"), list(identity.items()))

Identity,Observed value
notebook_path,/Users/lxg/Documents/GitHub/Alpha Research/alpha-research-lab-private/examples/research-foundations.ipynb
attempt_id,4549b993-5206-4cdf-84b0-9c29b2b73d92
commit,df2831e3e048d3d2a512690e7e010e9acd2df788
branch,codex/mr-08-saved-archive-rows
working_tree,changed
python,3.14.7
platform,macOS-27.0-arm64-arm-64bit-Mach-O
kernel,/Users/lxg/.cache/uv/builds-v0/.tmpfli9b9/bin/python
package,/Users/lxg/Documents/GitHub/Alpha Research/alpha-research-lab-private/.venv/lib/python3.14/site-packages/alpha_research
verified_runtime_files,59


## 2. Which saved samples will I inspect?

Set `ALPHA_OKX_ARCHIVE_DIRECTORY` and `ALPHA_BITFINEX_CAPTURE_DIRECTORY` to the
directories printed by the capture notebook to repeat a particular observation.
Without overrides, this exploration selects the **most recently saved completed
input with the exact maintained configuration**, by manifest save time. That
convenience selection can change after a new acquisition. Pin the printed paths
when a result matters; this is not an accepted research dataset.

The selection table records the paths, fingerprints and original implementation.
The next cells verify the bytes. A selected invalid input fails visibly; it never
triggers a replacement download or fallback to an older sample. No matching input
means run the capture notebook first.


In [3]:
codec = DomainCodec()
okx_config = OKXHistoryConfig.read(root / "configs/history/okx-btc-usdt.json")
bitfinex_config = CaptureConfig.read(root / "configs/live/bitfinex-btc-usd.json")
selected = {}
selection_rows = []
for venue, folder, variable, request_type, config in (
    ("okx", "data/okx/history", "ALPHA_OKX_ARCHIVE_DIRECTORY", OKXArchiveRequest, okx_config),
    (
        "bitfinex",
        "data/bitfinex",
        "ALPHA_BITFINEX_CAPTURE_DIRECTORY",
        CaptureRequest,
        bitfinex_config,
    ),
):
    override = os.environ.get(variable)
    candidates = []
    paths = (
        [Path(override).expanduser()]
        if override
        else [p.parent for p in (root / folder).glob("*/manifest.json")]
    )
    for path in paths:
        request = codec.decode_json((path / "request.json").read_bytes(), request_type)
        if request.config != config:
            if override:
                raise ValueError(f"{variable} differs from the maintained selection")
            continue
        descriptor = codec.decode_json((path / "manifest.json").read_bytes(), RawObject)
        candidates.append((descriptor.ts_ingested, str(path.resolve()), request, descriptor))
    if not candidates:
        raise FileNotFoundError(f"No matching {venue} input; run market-data-capture.ipynb first")
    _, location, request, descriptor = max(candidates, key=lambda item: (item[0], item[1]))
    selected[venue] = (Path(location), request, descriptor)
    selection_rows.extend(
        [
            (
                venue,
                "Selection",
                "explicit directory" if override else "latest matching saved input",
            ),
            (venue, "Directory", location),
            (venue, "Raw SHA-256", descriptor.sha256),
            (venue, "Request identity", request.identity),
            (venue, "Original implementation SHA-256", request.implementation_identity),
        ]
    )
NotebookDisplay.show_table(("Source", "Evidence", "Value"), selection_rows)

Source,Evidence,Value
okx,Selection,explicit directory
okx,Directory,/Users/lxg/Documents/GitHub/Alpha Research/alpha-research-lab-private/data/okx/history/802967bb-05eb-4b7b-9c90-2c964f22e41f
okx,Raw SHA-256,fef59b343d1972c32f67583edc538c5d2f7350e6f3a0059e6f87896d7e8bf676
okx,Request identity,71672efd00a8f3f6c959d0a08f8ce2a8793b3fb84d23f1a3c22b2756a6ec2665
okx,Original implementation SHA-256,22c0e38d88691bd0b1141588260465c471bbde0ff432a6bd1d0dab0505000014
bitfinex,Selection,explicit directory
bitfinex,Directory,/Users/lxg/Documents/GitHub/Alpha Research/alpha-research-lab-private/data/bitfinex/a5116c5a-b7d6-4e2b-9ee6-c79839c80f43
bitfinex,Raw SHA-256,a57ddabc5b1283c702ee2c7fd104a2b9cd943e2b41661cfc1d66619b35930163
bitfinex,Request identity,0e1453a2e7e925ad5bc1454a6a7d1e7298eca46e82a0d07b10e7ad8fc1c9ec21
bitfinex,Original implementation SHA-256,22c0e38d88691bd0b1141588260465c471bbde0ff432a6bd1d0dab0505000014


## 3. How much of the historical OKX day did I reconstruct?

The configured date is **8 September 2026**. The reader verifies the original
archive, then reconstructs at most **10,000 records**. The displayed source-time
interval is the processed prefix, not the whole day. L2 combines all reported
size at each price; it supplies no individual order identity or queue position.


In [4]:
okx_directory, okx_request, okx_descriptor = selected["okx"]
history = OKXArchiveStore(okx_directory).replay()
assert history["source_sequence"] is None and history["gaps"] is None
epoch = datetime(1970, 1, 1, tzinfo=UTC)
history_rows = [
    (key, history[key])
    for key in (
        "symbol",
        "representation",
        "records",
        "snapshots",
        "updates",
        "levels",
        "depth",
        "state_available",
        "continuity",
        "stop_reason",
        "entire_archive_replayed",
    )
]
for key in ("first_source_ms", "last_source_ms"):
    value = history[key]
    history_rows.append(
        (
            key.replace("_ms", "_utc"),
            None if value is None else epoch + timedelta(milliseconds=int(value)),
        )
    )
NotebookDisplay.show_table(("Historical observation", "Result"), history_rows)

Historical observation,Result
symbol,BTC-USDT
representation,L2
records,10000
snapshots,1
updates,9999
levels,800
depth,400
state_available,True
continuity,unproven_no_source_sequences
stop_reason,record_limit


**Read the result:** `gaps=None` means continuity is unknown: this archive has no
source sequence numbers. `state_available=True` means a book could be built from
the observed prefix; it does not certify uninterrupted historical coverage.
The archive API currently returns a reconstruction summary, not a research table
of historical prices or features.


## 4. What did the Bitfinex recording actually observe?

This is a saved bounded live recording, not downloadable Bitfinex historical data.
The table reports local receipt times and observed gaps. The R0 subscription
requests depth 25; the final number of retained visible orders is reported
separately. Sequence continuity covers this recording only.


In [5]:
bitfinex_directory, bitfinex_request, bitfinex_descriptor = selected["bitfinex"]
replay = CaptureReplay.read(bitfinex_directory)
summary = replay.summary()
assert isinstance(replay.book, BitfinexBook)
live_rows = [
    (key, summary[key])
    for key in (
        "symbol",
        "messages",
        "snapshots",
        "updates",
        "orders",
        "depth",
        "gaps",
        "synchronized",
        "source_sequence",
        "last_source_time",
    )
]
for key in ("started_ns", "finished_ns"):
    live_rows.append(
        (key.replace("_ns", "_utc"), epoch + timedelta(microseconds=summary[key] // 1000))
    )
NotebookDisplay.show_table(("Recorded observation", "Result"), live_rows)
assert summary["synchronized"] and not summary["gaps"], (
    "This sample is unsuitable for the uninterrupted-book preview; inspect its gaps"
)

Recorded observation,Result
symbol,tBTCUSD
messages,376
snapshots,1
updates,371
orders,50
depth,25
gaps,[]
synchronized,True
source_sequence,373
last_source_time,None


## 5. What visible prices and sizes can I inspect?

Read a few individual Bitfinex orders nearest each side's best price. Prices are
**USD per BTC**, sizes are **BTC**, and the displayed spread is **USD per BTC**.
Decimal values remain exact and are rendered as text. The spread is a descriptive
observation, not a trading return; it excludes fees, impact and fill uncertainty.


In [6]:
preview_rows = []
best = {}
for side in ("buy", "sell"):
    orders = sorted(
        (order for order in replay.book.orders.values() if order.side == side),
        key=lambda order: order.price,
        reverse=side == "buy",
    )
    assert orders, f"No observed {side} side"
    best[side] = orders[0].price
    for order in orders[:3]:
        assert isinstance(order.price, Decimal) and isinstance(order.size, Decimal)
        preview_rows.append((side, order.order_id, str(order.price), str(order.size)))
NotebookDisplay.show_table(("Side", "Visible order ID", "USD / BTC", "BTC"), preview_rows)
NotebookDisplay.show_table(
    ("Measure", "USD / BTC"),
    [
        ("Best bid", best["buy"]),
        ("Best ask", best["sell"]),
        ("Observed spread", best["sell"] - best["buy"]),
    ],
)

Side,Visible order ID,USD / BTC,BTC
buy,244232632883,75645,0.00180442
buy,244230114777,75639,0.0002
buy,244218842321,75638,0.0002
sell,244216587539,75655,0.00022
sell,244232116576,75660,0.03522185
sell,244217678512,75660,0.00022


Measure,USD / BTC
Best bid,75645
Best ask,75655
Observed spread,10


## 6. When did the market event happen, and when did I know about it?

OKX historical records have source timestamps, but their download time is much
later and there is no historical receipt clock. Bitfinex R0 has local receipt order
and sequences, but no exchange timestamp in these book messages. Neither clock
can fill in the other's missing information. UTC display truncates nanoseconds to
microseconds; original Bitfinex nanoseconds remain in the journal.


In [7]:
NotebookDisplay.show_table(
    ("Sample", "Source clock", "Local receipt evidence", "Saved bytes"),
    [
        (
            "OKX historical BTC-USDT",
            "Unix milliseconds in records; continuity unknown",
            f"Archive received {history['download_received']}; per-record receipt absent",
            history["saved_bytes"],
        ),
        (
            "Bitfinex recorded BTC/USD",
            "Absent in R0 messages",
            f"Journal nanoseconds {summary['started_ns']} to {summary['finished_ns']}",
            bitfinex_descriptor.size_bytes,
        ),
    ],
)

Sample,Source clock,Local receipt evidence,Saved bytes
OKX historical BTC-USDT,Unix milliseconds in records; continuity unknown,Archive received 2026-09-16T08:49:14.475436+00:00; per-record receipt absent,109423187
Bitfinex recorded BTC/USD,Absent in R0 messages,Journal nanoseconds 1789548508888411000 to 1789548529076738000,52908


## 7. What prevents me from analysing damaged or disconnected input?

These **controlled failures** modify temporary copies only: remove saved input,
change a saved byte, alter archive discovery, and inject a disconnect into a
separate reconstruction. The original recordings stay intact. Inspect the explicit
rejections and the unavailable book after the gap. A fresh subscription/snapshot
is needed for recovery; this example does not fill the gap with stale quotes.


In [8]:
# standard library imports
import shutil
from tempfile import TemporaryDirectory

failure_rows = []
with TemporaryDirectory(prefix="orderbook-input-checks-") as temporary:
    copy = Path(temporary)
    for name in ("request.json", "manifest.json"):
        shutil.copyfile(bitfinex_directory / name, copy / name)
    raw = copy / bitfinex_descriptor.sha256
    for label, expected in (
        ("Missing Bitfinex bytes", FileNotFoundError),
        ("Changed Bitfinex bytes", ValueError),
    ):
        if expected is ValueError:
            content = (bitfinex_directory / bitfinex_descriptor.sha256).read_bytes()
            raw.write_bytes(bytes([content[0] ^ 1]) + content[1:])
        try:
            CaptureReplay.read(copy)
        except expected as error:
            failure_rows.append((label, type(error).__name__))
        else:
            raise AssertionError(f"Accepted {label}")
    shutil.copyfile(okx_directory / "request.json", copy / "request.json")
    (copy / "index.json").write_bytes(b"{}")
    try:
        OKXArchiveStore(copy).replay()
    except ValueError as error:
        failure_rows.append(("Changed OKX discovery", str(error)))
    else:
        raise AssertionError("Accepted changed OKX discovery")

exercise = CaptureReplay(bitfinex_config)
for record in CaptureJournal(bitfinex_directory, bitfinex_request).records():
    if record.kind == "end":
        break
    exercise.apply(record)
exercise.apply(
    CaptureRecord(
        kind="gap",
        epoch=exercise.epoch,
        received_ns=exercise.finished_ns,
        reason="synthetic_disconnect",
    )
)
assert not exercise.book.ready
failure_rows.append(
    ("Injected disconnect", f"Book usable: {exercise.book.ready}; {exercise.gaps[-1]['reason']}")
)
NotebookDisplay.show_table(("Controlled case", "Observed result"), failure_rows)

Controlled case,Observed result
Missing Bitfinex bytes,FileNotFoundError
Changed Bitfinex bytes,ValueError
Changed OKX discovery,saved OKX archive discovery differs from request
Injected disconnect,Book usable: False; synthetic_disconnect


## 8. What can I conclude, and what should I investigate next?

The last check reopens the Bitfinex input and compares the entire sequence of
reconstructed states. This establishes repeatability from the same saved bytes
and installed code, not data completeness or executable fills.


In [9]:
again = CaptureReplay.read(bitfinex_directory)
assert again.summary()["trace_identity"] == summary["trace_identity"]
NotebookDisplay.show_table(
    ("Research finding", "Evidence or limit"),
    [
        (
            "Historical liquidity observations",
            f"OKX: {history['records']} prefix records, {history['levels']} final levels",
        ),
        (
            "Recorded visible orders",
            f"Bitfinex: {summary['messages']} messages, {summary['orders']} final orders",
        ),
        ("Repeatability", f"Matching Bitfinex state trace: {summary['trace_identity']}"),
        ("Controlled failures", f"{len(failure_rows)} explicit outcomes"),
        (
            "Not established",
            "Same-time prices, full coverage, historical eligibility, queue priority or fills",
        ),
        (
            "Next roadmap MR",
            "MR 09: bind decoded historical events to raw observations; not implemented here",
        ),
    ],
)

Research finding,Evidence or limit
Historical liquidity observations,"OKX: 10000 prefix records, 800 final levels"
Recorded visible orders,"Bitfinex: 376 messages, 50 final orders"
Repeatability,Matching Bitfinex state trace: 283533b937763c40d108517fc22ddbbc2d10a77820a8de498cc631f37c11d375
Controlled failures,4 explicit outcomes
Not established,"Same-time prices, full coverage, historical eligibility, queue priority or fills"
Next roadmap MR,MR 09: bind decoded historical events to raw observations; not implemented here


For a new observation, deliberately run the [capture notebook](market-data-capture.ipynb)
and pin its printed directories before rerunning this study. For timing, exact
financial units and saved-row boundary exercises, use the independent
[offline reliability lab](foundation-contract-checks.ipynb). Keep an influencing
trial in the research journal; these local snapshots are not its backup.
